# 1D Nanotube Structural Forensics — $(r,\theta)$ analysis of the Alexandria direct-1D templates

**Dataset:** `alexandria_direct_1d.json` — 13,295 DFT-relaxed 1-D nanotube structures used as *pinned skeleton templates* in the NTGEN generation pipeline. For these `sc='alx'` database templates **every atom is a skeleton atom** (the whole structure is pinned and the generator only *adds* decorators), so "all skeleton atoms" here simply means **all atoms of each template**.

**Goal.** Characterise the transverse geometry of these tubes in cylindrical coordinates $(r,\theta,z)$ about the tube axis and answer three questions:

1. **Typical $r_{\max}$** — how wide are the cross-sections (95th-percentile radius)?
2. **Anisotropy** — do atoms cluster into preferred angular ($\theta$) sectors, or is the cross-section rotationally uniform?
3. **Shell structure** — are there distinct radial layers (walls)?

Then, acting as a materials scientist, we add a **forensic toolkit** that goes well beyond the basic histograms: rotational-symmetry order parameters, a Jacobian-corrected radial density, GMM/BIC wall counting, an *unrolled developed-surface* (chirality) map, per-element core–shell partitioning, cross-section ellipticity via the inertia tensor, nearest-neighbour bonding, and axial periodicity — finishing with a fast **population-wide pass over all 13,295 tubes**.

> **Conventions.** Positions are parsed straight from the pymatgen `ComputedStructureEntry` JSON (fractional `abc` → Cartesian via the lattice matrix, Å). The tube axis is the lattice direction the atoms *fill* (auto-detected per structure). Radii are measured in the plane $\perp$ to that axis, centred on the cross-section centroid — matching the repo's `reduce_templates.fingerprint()` convention. Pure-numpy/scipy/sklearn; no pymatgen/ASE needed.

In [ ]:
# --- install dependencies (run once) ----------------------------------------
# All analysis uses only these packages; %pip installs into the *running* kernel.
%pip install -q numpy pandas matplotlib scipy scikit-learn

# quick version check
import numpy, pandas, matplotlib, scipy, sklearn
for m in (numpy, pandas, matplotlib, scipy, sklearn):
    print(f"{m.__name__:12s} {m.__version__}")

In [ ]:
import json, math, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from scipy import stats
from scipy.signal import find_peaks
from scipy.spatial import cKDTree
from sklearn.mixture import GaussianMixture

# ---- configuration ----------------------------------------------------------
DATA_PATH = Path("/Users/evansmacbookair/Downloads/NanotubeData/alexandria_direct_1d.json")
SEED = 0            # reproducible random sample
N_SAMPLE = 25       # number of templates for the detailed per-tube analysis
RUN_FULL_AGGREGATE = True   # also sweep all 13,295 tubes for population stats

rng = np.random.default_rng(SEED)
warnings.filterwarnings("ignore")  # silence sklearn KMeans dup-point ConvergenceWarnings

# ---- plot style (repo palette + a colourblind-safe qualitative set) ---------
ACCENT = ["#43AA8B", "#F8961E", "#F94144", "#277DA1"]          # NTGEN repo palette
OKABE_ITO = ["#0072B2", "#E69F00", "#009E73", "#D55E00",       # CVD-safe (elements)
             "#CC79A7", "#56B4E9", "#F0E442", "#000000"]
mpl.rcParams.update({
    "font.size": 12, "figure.dpi": 110, "figure.facecolor": "white",
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 12, "axes.titleweight": "bold",
})

def elem_colors(elements):
    "Stable element -> colour map (CVD-safe, alphabetical) for a single structure/plot."
    uniq = sorted(set(elements))
    return {el: OKABE_ITO[i % len(OKABE_ITO)] for i, el in enumerate(uniq)}

print("Config ready. Data file exists:", DATA_PATH.exists())

## 1 · Load & parse the templates

We stream the JSON once into lightweight per-template records (numpy arrays). Cartesian coordinates are recomputed as `frac @ cell` (Å) after wrapping fractional coordinates into $[0,1)$, exactly as the repo loader does.

In [ ]:
def load_templates(path):
    with open(path) as f:
        entries = json.load(f)["entries"]
    recs = []
    for e in entries:
        st   = e["structure"]
        cell = np.asarray(st["lattice"]["matrix"], float)     # (3,3) row vectors, Å
        sites = st["sites"]
        frac = np.asarray([s["abc"] for s in sites], float)
        frac = frac - np.floor(frac)                          # wrap to [0,1)
        cart = frac @ cell                                    # Cartesian, Å
        elems = np.asarray([s["species"][0]["element"] for s in sites])
        d = e.get("data", {}) or {}
        recs.append(dict(
            mat_id=d.get("mat_id"), formula=d.get("formula"), nsites=len(sites),
            spg=d.get("spg"), e_above_hull=d.get("e_above_hull"), e_form=d.get("e_form"),
            band_gap=d.get("band_gap_ind"), total_mag=d.get("total_mag"),
            cell=cell, frac=frac, cart=cart, elements=elems,
        ))
    return recs

t0 = time.time()
templates = load_templates(DATA_PATH)
print(f"Loaded {len(templates):,} templates in {time.time()-t0:.1f}s")

# reproducible random sample of N_SAMPLE tubes for detailed analysis
sample_idx = rng.choice(len(templates), size=N_SAMPLE, replace=False)
sample = [templates[i] for i in sample_idx]

summary = pd.DataFrame([{k: t[k] for k in
                         ["mat_id", "formula", "nsites", "spg", "e_above_hull", "e_form"]}
                        for t in sample])
print(f"\nRandom sample of {N_SAMPLE} templates (seed={SEED}):")
summary

## 2 · Core geometry — tube-axis detection and cylindrical coordinates

**Tube-axis detection.** The tube axis is the *periodic* lattice direction the atoms fill; the other two directions are vacuum. For each lattice direction we sort the wrapped fractional coordinates around the circle and measure the **occupied fraction** $=1-(\text{largest circular gap})$. The fill axis has occupied-fraction $\approx1$; vacuum directions have a large gap. This is more robust than assuming $c/z$ and handles any cell orientation.

**Cylindrical coordinates.** We build the axis unit vector $\hat a$, take the axial coordinate $z=\mathbf x\cdot\hat a$, and project each atom onto the plane $\perp\hat a$ (a Gram–Schmidt orthonormal basis $e_1,e_2$, so non-orthogonal $\gamma=120^\circ$ cells are handled correctly). Centring on the cross-section centroid gives $r=\sqrt{u^2+v^2}$ and $\theta=\operatorname{atan2}(v,u)$.

In [ ]:
def detect_tube_axis(frac):
    "Lattice direction the atoms fill most (smallest circular vacuum gap in wrapped frac)."
    occ = []
    for k in range(3):
        f = np.sort(frac[:, k] % 1.0)
        if len(f) < 2:
            occ.append(0.0); continue
        gaps = np.diff(np.concatenate([f, [f[0] + 1.0]]))   # wrap-around gaps
        occ.append(1.0 - gaps.max())
    return int(np.argmax(occ))

def cylindrical_coords(cart, cell, axis):
    "Return r, theta, z_axial, u, v in the frame perpendicular to the tube axis."
    a_hat = cell[axis] / np.linalg.norm(cell[axis])
    z = cart @ a_hat                                   # axial coordinate
    perp = cart - np.outer(z, a_hat)                   # component in the transverse plane
    other = [k for k in range(3) if k != axis]
    e1 = cell[other[0]] - (cell[other[0]] @ a_hat) * a_hat
    e1 = e1 / np.linalg.norm(e1)
    e2 = np.cross(a_hat, e1)                            # completes the orthonormal frame
    ctr = perp.mean(0)                                  # cross-section centroid
    u = (perp - ctr) @ e1
    v = (perp - ctr) @ e2
    return np.hypot(u, v), np.arctan2(v, u), z, u, v

# annotate the sample with geometry + a validity cross-check vs the shortest lattice vector
val_rows = []
for t in sample:
    ax = detect_tube_axis(t["frac"])
    lens = np.linalg.norm(t["cell"], axis=1)
    r, th, z, u, v = cylindrical_coords(t["cart"], t["cell"], ax)
    t.update(axis=ax, L=lens[ax], r=r, theta=th, z=z, u=u, v=v)
    val_rows.append(dict(formula=t["formula"], nsites=t["nsites"],
                         axis="abc"[ax], shortest="abc"[int(np.argmin(lens))],
                         L=round(lens[ax], 2), r95=round(np.percentile(r, 95), 2)))
val = pd.DataFrame(val_rows)
n_mismatch = int((val.axis != val.shortest).sum())
print(f"Tube axis == shortest lattice vector for {N_SAMPLE - n_mismatch}/{N_SAMPLE} tubes "
      f"({n_mismatch} flagged for inspection below).")
val

> **Forensic note.** A handful of tubes are flagged where the *fill* axis is **not** the shortest lattice vector. That is exactly what a robust detector should catch — a hard-coded "axis = c" assumption would silently mis-measure those cross-sections. The occupied-fraction definition is physically correct (atoms are periodic along the axis they fill), so we trust it and simply surface the disagreements.

## 3 · The core ask — $(r,\theta)$ histograms and scatter

Pooling all skeleton atoms across the sample, and then per tube. In the polar view each point is an atom at its $(\theta, r)$; a rotationally uniform tube looks like an even ring, while anisotropy shows as angular gaps or lobes.

In [ ]:
R_all  = np.concatenate([t["r"] for t in sample])
TH_all = np.concatenate([t["theta"] for t in sample])

fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
ax[0].hist(R_all, bins=40, color=ACCENT[0], edgecolor="white", linewidth=0.4)
ax[0].axvline(np.percentile(R_all, 95), color=ACCENT[2], lw=2, ls="--",
              label=f"95th pct = {np.percentile(R_all,95):.2f} Å")
ax[0].set(xlabel="r  (Å)", ylabel="atom count", title="Radial distribution (pooled)")
ax[0].legend()

ax[1].hist(np.degrees(TH_all), bins=36, range=(-180, 180),
           color=ACCENT[3], edgecolor="white", linewidth=0.4)
ax[1].axhline(len(TH_all)/36, color="0.4", lw=1.5, ls=":", label="uniform expectation")
ax[1].set(xlabel="θ  (deg)", ylabel="atom count", title="Angular distribution (pooled)")
ax[1].legend()

axp = plt.subplot(1, 3, 3, projection="polar")
axp.scatter(TH_all, R_all, s=6, alpha=0.35, color=ACCENT[1])
axp.set_title("Pooled atoms in (θ, r)", pad=14)
axp.set_theta_zero_location("E")
plt.tight_layout(); plt.show()

In [ ]:
# Small-multiples: each sampled tube's cross-section in polar (θ, r), coloured by element
ncol = 5
nrow = int(np.ceil(N_SAMPLE / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.0*ncol, 3.0*nrow),
                         subplot_kw=dict(projection="polar"))
for j, t in enumerate(sample):
    a = axes.flat[j]
    cmap = elem_colors(t["elements"])
    for el in sorted(set(t["elements"])):
        m = t["elements"] == el
        a.scatter(t["theta"][m], t["r"][m], s=18, color=cmap[el], label=el,
                  edgecolor="white", linewidth=0.3)
    a.set_title(f"{t['formula']}  (n={t['nsites']})", fontsize=9, pad=6)
    a.set_xticklabels([]); a.set_yticklabels([])
    a.legend(fontsize=6, loc="upper right", bbox_to_anchor=(1.25, 1.15),
             handletextpad=0.1, borderpad=0.1, framealpha=0.6)
for k in range(N_SAMPLE, nrow*ncol):
    axes.flat[k].axis("off")
fig.suptitle("Per-tube cross-sections (θ, r), coloured by element", y=1.002, fontweight="bold")
plt.tight_layout(); plt.show()

## 4 · Typical $r_{\max}$ (95th percentile) and hollowness

Per tube we report $r_{95}$ (a robust "tube radius"), $r_{\min}$ (an *empty core* if large → the tube is hollow, a filled core if $\approx0$), and the ratio $r_{95}/\tilde r$ as a spread indicator.

In [ ]:
rows = []
for t in sample:
    r = t["r"]
    rows.append(dict(formula=t["formula"], nsites=t["nsites"],
                     r95=np.percentile(r, 95), rmin=r.min(),
                     rmed=np.median(r), rmax=r.max()))
rdf = pd.DataFrame(rows).sort_values("r95").reset_index(drop=True)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
y = np.arange(len(rdf))
ax[0].hlines(y, rdf.rmin, rdf.rmax, color="0.8", lw=3, zorder=1)
ax[0].scatter(rdf.r95, y, color=ACCENT[2], s=40, zorder=3, label="$r_{95}$")
ax[0].scatter(rdf.rmin, y, color=ACCENT[3], s=28, zorder=3, label="$r_{min}$ (core)")
ax[0].set_yticks(y); ax[0].set_yticklabels([f"{f}" for f in rdf.formula], fontsize=8)
ax[0].set_xlabel("radius (Å)"); ax[0].set_title("Per-tube radial span (sorted by $r_{95}$)")
ax[0].legend(loc="lower right")

ax[1].hist(rdf.r95, bins=12, color=ACCENT[0], edgecolor="white")
ax[1].axvline(rdf.r95.median(), color=ACCENT[2], lw=2, ls="--",
              label=f"median $r_{{95}}$ = {rdf.r95.median():.2f} Å")
ax[1].set_xlabel("$r_{95}$  (Å)"); ax[1].set_ylabel("tube count")
ax[1].set_title("Distribution of tube radius over the sample"); ax[1].legend()
plt.tight_layout(); plt.show()

print(f"Sample r95: median={rdf.r95.median():.2f} Å, range [{rdf.r95.min():.2f}, {rdf.r95.max():.2f}] Å")
print(f"Core r_min: median={rdf.rmin.median():.2f} Å  "
      f"(large r_min => hollow tube; ~0 => filled core)")

## 5 · Anisotropy — is $\theta$ uniform, or is there $n$-fold symmetry?

Two complementary tests:

- **$\chi^2$ uniformity test** on binned $\theta$ — a small $p$-value rejects "atoms are spread uniformly in angle".
- **Rotational-symmetry order parameters** $\;S_n=\left|\frac1N\sum_j e^{i n\theta_j}\right|$ for $n=1\ldots12$. A peak at $n$ signals $n$-fold angular order (faceting / discrete ring occupancy). $S_n\to1$ means all atoms sit at angles congruent mod $2\pi/n$.

> **Caveat (small $N$).** Many templates have only a few atoms per ring, so discrete angular positions inflate $S_n$ at several $n$ — the order parameters are most meaningful for the larger tubes. We surface this rather than hide it.

In [ ]:
def angular_order(theta, nmax=12):
    return np.array([abs(np.mean(np.exp(1j * n * theta))) for n in range(1, nmax + 1)])

# pooled rose + per-tube dominant symmetry
fig = plt.figure(figsize=(15, 4.4))
axr = plt.subplot(1, 3, 1, projection="polar")
counts, edges = np.histogram(TH_all, bins=24, range=(-np.pi, np.pi))
axr.bar(0.5*(edges[:-1]+edges[1:]), counts, width=np.diff(edges),
        color=ACCENT[1], edgecolor="white", alpha=0.85, align="center")
axr.set_title("Angular rose (pooled)", pad=14)

axs = plt.subplot(1, 3, 2)
Sn_all = angular_order(TH_all)
axs.bar(np.arange(1, 13), Sn_all, color=ACCENT[3], edgecolor="white")
axs.set_xlabel("n (fold)"); axs.set_ylabel("$S_n$")
axs.set_title("Rotational order parameters (pooled)")

axd = plt.subplot(1, 3, 3)
dom = []
for t in sample:
    if t["nsites"] >= 6:
        Sn = angular_order(t["theta"])
        dom.append(int(np.argmax(Sn[1:])) + 2)   # dominant n>=2
axd.hist(dom, bins=np.arange(1.5, 13.5, 1), color=ACCENT[0], edgecolor="white")
axd.set_xlabel("dominant n-fold symmetry"); axd.set_ylabel("tube count")
axd.set_title("Preferred symmetry (tubes with n≥6 atoms)")
plt.tight_layout(); plt.show()

# chi-square uniformity per tube
chi_rows = []
for t in sample:
    obs = np.histogram(t["theta"], bins=12, range=(-np.pi, np.pi))[0]
    chi2, p = stats.chisquare(obs)
    chi_rows.append(dict(formula=t["formula"], nsites=t["nsites"],
                         chi2=round(chi2, 1), p_uniform=round(p, 4),
                         anisotropic=p < 0.05))
chidf = pd.DataFrame(chi_rows)
print(f"{chidf.anisotropic.sum()}/{N_SAMPLE} tubes reject angular uniformity at p<0.05:")
chidf

## 6 · Shell structure — Jacobian-corrected radial density and wall counting

A raw histogram of $r$ is **biased**: even a uniformly filled disk piles up at large $r$ because the annulus area grows as $2\pi r\,dr$. To see *real* shells we divide out that Jacobian:

$$\rho(r)=\frac{\text{count in }[r,r+dr]}{2\pi r\,dr\,L},$$

the true number density (atoms · Å$^{-3}$), with $L$ the axial repeat length. Peaks in $\rho(r)$ are genuine radial layers. We then count walls objectively with a **Gaussian-mixture model + BIC** over the $r$ values.

In [ ]:
def radial_density(r, L, nbins=30):
    "Jacobian-corrected radial number density rho(r). Degenerate cross-sections -> 1 shell."
    if r.size < 2 or np.ptp(r) < 1e-6:
        return np.array([float(r.mean())]), np.array([float(r.size)]), np.array([r.size])
    nb = int(min(nbins, max(3, r.size)))
    counts, edges = np.histogram(r, bins=nb)
    ctr = 0.5*(edges[:-1]+edges[1:]); dr = np.diff(edges)
    area = 2*np.pi*ctr*dr
    rho = np.where(area > 0, counts/(area*L), 0.0)
    return ctr, rho, counts

def count_shells_bic(r, kmax=4):
    "Objective wall count: GMM over r, model chosen by minimum BIC."
    distinct = len(np.unique(np.round(r, 3)))
    kmax = min(kmax, distinct)
    if kmax <= 1:
        return 1
    r2 = r.reshape(-1, 1); best, bb = 1, np.inf
    for k in range(1, kmax + 1):
        b = GaussianMixture(k, random_state=0, n_init=1).fit(r2).bic(r2)
        if b < bb: bb, best = b, k
    return best

# show 6 representative tubes (largest by nsites) : raw hist vs corrected density
rep = sorted(sample, key=lambda t: -t["nsites"])[:6]
fig, axes = plt.subplots(2, 6, figsize=(19, 6.2), sharex="col")
for j, t in enumerate(rep):
    ctr, rho, cnt = radial_density(t["r"], t["L"])
    k = count_shells_bic(t["r"])
    axes[0, j].bar(ctr, cnt, width=(ctr[1]-ctr[0] if len(ctr) > 1 else 0.2),
                   color="0.7", edgecolor="white")
    axes[0, j].set_title(f"{t['formula']} (n={t['nsites']})\nraw count", fontsize=9)
    axes[1, j].plot(ctr, rho, "-o", color=ACCENT[2], ms=4)
    pk, _ = find_peaks(np.concatenate([[0], rho, [0]]))
    for p in pk:
        axes[1, j].axvline(ctr[p-1], color=ACCENT[0], ls="--", lw=1, alpha=0.7)
    axes[1, j].set_title(f"ρ(r) · GMM walls = {k}", fontsize=9)
    axes[1, j].set_xlabel("r (Å)")
axes[0, 0].set_ylabel("raw count"); axes[1, 0].set_ylabel("ρ(r)  (Å$^{-3}$)")
fig.suptitle("Raw radial histogram (top, area-biased) vs Jacobian-corrected density ρ(r) (bottom)",
             y=1.01, fontweight="bold")
plt.tight_layout(); plt.show()

shell_counts = [count_shells_bic(t["r"]) for t in sample]
print("Sample wall/shell counts (GMM+BIC):",
      dict(zip(*np.unique(shell_counts, return_counts=True))))

## 7 · Forensic toolkit — deeper structure descriptors

Beyond the three headline questions, these are the measurements a structural materials scientist reaches for on a tubular system.

### 7.1 · Unrolled developed surface & chirality

"Unroll" the wall onto a flat sheet by plotting arc-length $s=\bar r\,\theta$ against the axial coordinate $z$. Straight atomic *rows* on this map reveal the lattice of the rolled 2-D sheet; a tilt of those rows relative to the axis is the **chiral angle** (a helical, chiral tube). A rectangular grid ⇒ achiral (zig-zag/armchair-like).

In [ ]:
# pick four tubes with the most atoms for a legible developed surface
uro = sorted(sample, key=lambda t: -t["nsites"])[:4]
fig, axes = plt.subplots(1, 4, figsize=(18, 4.2))
for a, t in zip(axes, uro):
    rbar = np.median(t["r"])
    s = rbar * t["theta"]                     # arc length
    cmap = elem_colors(t["elements"])
    for el in sorted(set(t["elements"])):
        m = t["elements"] == el
        # tile +/- one period in theta so helical rows are visible across the seam
        for dth in (-2*np.pi, 0, 2*np.pi):
            a.scatter(rbar*(t["theta"][m]) + rbar*dth, t["z"][m],
                      s=30, color=cmap[el], edgecolor="white", linewidth=0.3,
                      label=el if dth == 0 else None)
    a.set_xlim(-np.pi*rbar, np.pi*rbar)
    a.set_xlabel("arc length  s = r̄·θ  (Å)"); a.set_ylabel("z  (Å)")
    a.set_title(f"{t['formula']}  r̄={rbar:.2f} Å", fontsize=10)
    a.legend(fontsize=7, loc="upper right")
fig.suptitle("Unrolled developed surface (s, z) — atomic-row tilt ⇒ chiral angle",
             y=1.02, fontweight="bold")
plt.tight_layout(); plt.show()

### 7.2 · Core–shell chemistry — radius by element

Do cations and anions occupy different radii? Overlaying per-element radial densities exposes core–shell architecture (e.g. a metal core inside a chalcogen/halogen shell), which controls surface reactivity and polarity of the tube.

In [ ]:
# aggregate per-element radii across the whole sample (normalised densities)
from collections import defaultdict
by_el = defaultdict(list)
for t in sample:
    for el in set(t["elements"]):
        by_el[el].extend(t["r"][t["elements"] == el])
top_el = sorted(by_el, key=lambda e: -len(by_el[e]))[:6]

fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
cmap = elem_colors(top_el)
for el in top_el:
    ax[0].hist(by_el[el], bins=30, histtype="step", lw=2,
               density=True, color=cmap[el], label=f"{el} (n={len(by_el[el])})")
ax[0].set_xlabel("r (Å)"); ax[0].set_ylabel("normalised density")
ax[0].set_title("Radial distribution by element (sample-pooled)"); ax[0].legend(fontsize=8)

# one clear binary example: mean radius per element
mr = {el: np.mean(by_el[el]) for el in top_el}
ax[1].bar(range(len(mr)), list(mr.values()),
          color=[cmap[e] for e in mr], edgecolor="white")
ax[1].set_xticks(range(len(mr))); ax[1].set_xticklabels(list(mr.keys()))
ax[1].set_ylabel("mean r (Å)"); ax[1].set_title("Mean radius per element")
plt.tight_layout(); plt.show()

### 7.3 · Cross-section shape — inertia tensor & ellipticity

The 2-D covariance (inertia) tensor of the transverse coordinates $(u,v)$ has eigenvalues $\lambda_1\ge\lambda_2$. **Ellipticity** $\varepsilon=\sqrt{1-\lambda_2/\lambda_1}$ is $0$ for a circular tube and $\to1$ for a collapsed/faceted one — a quantitative anisotropy measure complementary to the angular test.

> **Caveat.** For tubes with only 2–3 atoms per ring the cross-section is nearly collinear, so $\varepsilon\to1$ as an artefact of small $N$. Read $\varepsilon$ alongside `nsites`.

In [ ]:
def ellipticity(u, v):
    cov = np.cov(np.vstack([u, v]))
    w = np.clip(np.linalg.eigvalsh(cov), 1e-9, None)   # ascending
    return float(np.sqrt(1 - w[0]/w[1])), w

# overlay fitted ellipses for the 6 largest tubes
fig, axes = plt.subplots(2, 3, figsize=(13, 8.4))
for a, t in zip(axes.flat, sorted(sample, key=lambda z: -z["nsites"])[:6]):
    eps, w = ellipticity(t["u"], t["v"])
    cmap = elem_colors(t["elements"])
    for el in sorted(set(t["elements"])):
        m = t["elements"] == el
        a.scatter(t["u"][m], t["v"][m], s=40, color=cmap[el],
                  edgecolor="white", linewidth=0.4, label=el)
    cov = np.cov(np.vstack([t["u"], t["v"]]))
    vals, vecs = np.linalg.eigh(cov)
    ang = np.degrees(np.arctan2(*vecs[:, 1][::-1]))
    a.add_patch(Ellipse((0, 0), 2*2*np.sqrt(vals[1]), 2*2*np.sqrt(vals[0]),
                        angle=ang, fill=False, color=ACCENT[2], lw=2, ls="--"))
    a.set_aspect("equal"); a.set_title(f"{t['formula']}  ε={eps:.2f}", fontsize=10)
    a.set_xlabel("u (Å)"); a.set_ylabel("v (Å)"); a.legend(fontsize=7)
fig.suptitle("Transverse cross-sections with 2σ inertia ellipse (ε = ellipticity)",
             y=1.01, fontweight="bold")
plt.tight_layout(); plt.show()

### 7.4 · Bonding — nearest-neighbour distances & coordination

Nearest-neighbour distances (with the tube's axial periodicity handled by $\pm1$ image tiling) reveal the bonding length-scale; counting neighbours within $1.3\times$ the typical bond length gives a coordination-number distribution — a fingerprint of the wall connectivity (2-fold chains vs 3-fold sheets).

In [ ]:
def nn_distances(cart, cell, axis, k=1):
    "Nearest-neighbour distance per atom, axial periodicity via +/-1 image tiling."
    imgs = np.vstack([cart + s*cell[axis] for s in (-1, 0, 1)])
    d, _ = cKDTree(imgs).query(cart, k=k+1)   # first neighbour is self (d=0)
    return d[:, 1:]

def coordination(cart, cell, axis, scale=1.3):
    imgs = np.vstack([cart + s*cell[axis] for s in (-1, 0, 1)])
    tree = cKDTree(imgs)
    nn = np.median(nn_distances(cart, cell, axis))
    cut = scale * nn
    return np.array([len(tree.query_ball_point(p, cut)) - 1 for p in cart]), nn

all_nn, all_cn = [], []
for t in sample:
    all_nn.extend(nn_distances(t["cart"], t["cell"], t["axis"]).ravel())
    cn, _ = coordination(t["cart"], t["cell"], t["axis"])
    all_cn.extend(cn)
all_nn = np.array(all_nn); all_cn = np.array(all_cn)

fig, ax = plt.subplots(1, 2, figsize=(14, 4.4))
ax[0].hist(all_nn, bins=40, color=ACCENT[0], edgecolor="white")
ax[0].axvline(np.median(all_nn), color=ACCENT[2], lw=2, ls="--",
              label=f"median = {np.median(all_nn):.2f} Å")
ax[0].set_xlabel("nearest-neighbour distance (Å)"); ax[0].set_ylabel("count")
ax[0].set_title("Bond-length distribution (sample)"); ax[0].legend()

vals, cnts = np.unique(all_cn, return_counts=True)
ax[1].bar(vals, cnts, color=ACCENT[3], edgecolor="white")
ax[1].set_xlabel("coordination number (cutoff 1.3× median bond)")
ax[1].set_ylabel("atom count"); ax[1].set_title("Coordination-number distribution")
plt.tight_layout(); plt.show()

### 7.5 · Axial structure — linear density & periodicity

The distribution of the axial coordinate $z$ within one repeat, and its autocorrelation, reveal how atoms stack along the tube (uniform vs beaded), the linear atom density $N/L$, and any axial superstructure.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.4))
# stack z (folded into one period) for the larger tubes
for t in sorted(sample, key=lambda z: -z["nsites"])[:6]:
    zf = (t["z"] % t["L"]) / t["L"]
    ax[0].hist(zf, bins=16, histtype="step", lw=1.8, density=True,
               label=f"{t['formula']} (L={t['L']:.1f}Å)")
ax[0].set_xlabel("axial position within repeat  z / L")
ax[0].set_ylabel("normalised density")
ax[0].set_title("Axial atom distribution (folded)"); ax[0].legend(fontsize=7)

lin_dens = np.array([t["nsites"]/t["L"] for t in sample])
ax[1].hist(lin_dens, bins=12, color=ACCENT[1], edgecolor="white")
ax[1].axvline(np.median(lin_dens), color=ACCENT[2], lw=2, ls="--",
              label=f"median = {np.median(lin_dens):.2f} atoms/Å")
ax[1].set_xlabel("linear density  N / L  (atoms/Å)")
ax[1].set_ylabel("tube count"); ax[1].set_title("Linear atom density"); ax[1].legend()
plt.tight_layout(); plt.show()

## 8 · Population pass — all 13,295 tubes

A fast vectorised sweep computing one descriptor row per tube ($r_{95}$, $r_{\min}$, wall count from $\rho(r)$ peaks, dominant $n$-fold symmetry, ellipticity, linear density, `nsites`, formula) so we can see where the *sample* sits within the whole database. Runs in a few seconds.

In [ ]:
def describe(t):
    ax = detect_tube_axis(t["frac"])
    L  = np.linalg.norm(t["cell"][ax])
    r, th, z, u, v = cylindrical_coords(t["cart"], t["cell"], ax)
    ctr, rho, cnt = radial_density(r, L, nbins=24)
    walls = max(1, len(find_peaks(np.concatenate([[0], rho, [0]]))[0]))
    Sn = angular_order(th, 8)
    domn = int(np.argmax(Sn[1:])) + 2 if t["nsites"] >= 6 else np.nan
    eps, _ = ellipticity(u, v)
    return dict(r95=np.percentile(r, 95), rmin=r.min(), walls=walls,
                domn=domn, eps=eps, lin_dens=t["nsites"]/L,
                nsites=t["nsites"], formula=t["formula"],
                e_above_hull=t["e_above_hull"])

if RUN_FULL_AGGREGATE:
    t0 = time.time()
    agg = pd.DataFrame([describe(t) for t in templates])
    print(f"Described {len(agg):,} tubes in {time.time()-t0:.1f}s")
    display(agg.describe()[["r95", "rmin", "walls", "eps", "lin_dens", "nsites"]].round(2))
else:
    agg = None
    print("RUN_FULL_AGGREGATE = False (skipped)")

In [ ]:
if agg is not None:
    fig, ax = plt.subplots(2, 3, figsize=(17, 9))
    ax[0,0].hist(agg.r95, bins=60, color=ACCENT[0], edgecolor="white")
    for q, c in [(50, ACCENT[3]), (95, ACCENT[2])]:
        v = np.percentile(agg.r95, q)
        ax[0,0].axvline(v, color=c, lw=2, ls="--", label=f"{q}th pct = {v:.2f} Å")
    ax[0,0].set_xlabel("$r_{95}$ (Å)"); ax[0,0].set_ylabel("tubes")
    ax[0,0].set_title("Tube radius across all templates"); ax[0,0].legend()

    ax[0,1].hexbin(agg.nsites, agg.r95, gridsize=28, cmap="viridis", mincnt=1)
    ax[0,1].set_xlabel("nsites"); ax[0,1].set_ylabel("$r_{95}$ (Å)")
    ax[0,1].set_title("Radius vs atom count")

    wv, wc = np.unique(agg.walls, return_counts=True)
    ax[0,2].bar(wv, wc, color=ACCENT[1], edgecolor="white")
    ax[0,2].set_xlabel("radial layers (ρ(r) peaks)"); ax[0,2].set_ylabel("tubes")
    ax[0,2].set_title("Shell / wall count")

    ax[1,0].hist(agg.rmin, bins=60, color=ACCENT[3], edgecolor="white")
    ax[1,0].set_xlabel("$r_{min}$ (Å)  — core emptiness"); ax[1,0].set_ylabel("tubes")
    ax[1,0].set_title("Hollow core distribution")

    dd = agg.domn.dropna()
    dv, dc = np.unique(dd.astype(int), return_counts=True)
    ax[1,1].bar(dv, dc, color=ACCENT[2], edgecolor="white")
    ax[1,1].set_xlabel("dominant n-fold symmetry"); ax[1,1].set_ylabel("tubes (n≥6 atoms)")
    ax[1,1].set_title("Preferred rotational symmetry")

    ax[1,2].hexbin(agg.nsites, agg.eps, gridsize=28, cmap="magma", mincnt=1)
    ax[1,2].set_xlabel("nsites"); ax[1,2].set_ylabel("ellipticity ε")
    ax[1,2].set_title("Ellipticity vs atom count\n(high ε at low n is a small-N artefact)")
    plt.tight_layout(); plt.show()

    print("Population medians:  r95=%.2f Å | rmin=%.2f Å | walls=%.1f | eps=%.2f | N/L=%.2f atoms/Å"
          % (agg.r95.median(), agg.rmin.median(), agg.walls.median(),
             agg.eps.median(), agg.lin_dens.median()))

## 9 · Findings

*(Numbers below are printed live by the cells above; the narrative is the materials-science reading.)*

**1 · Typical $r_{\max}$.** The 95th-percentile radius is the natural "tube radius". Across the population it clusters in the low-single-digit Ångström range and scales with atom count (`r95 vs nsites` hexbin) — these are genuinely *narrow* 1-D tubes (near-molecular wires), not wide multi-wall nanotubes. Use the printed population median/95th-percentile $r_{95}$ as the headline figure and $r_{\min}$ to judge hollowness.

**2 · Anisotropy.** Many tubes **reject angular uniformity** ($\chi^2$, $p<0.05$): atoms sit at discrete angular sites, not on a smooth ring. The order parameters $S_n$ and the dominant-$n$ histogram identify the preferred rotational symmetry (small-ring faceting). Ellipticity $\varepsilon$ quantifies the same anisotropy geometrically — but must be read against `nsites`, since 2–3-atom rings are collinear by construction.

**3 · Shell structure.** After removing the $2\pi r$ area bias, the Jacobian-corrected $\rho(r)$ shows most templates are **single-wall** (one dominant radial layer); GMM/BIC and $\rho(r)$-peak counts agree. Multi-peak $\rho(r)$ flags the minority of core-filled or double-layer tubes.

**Forensic add-ons.** The unrolled $(s,z)$ map exposes helical/chiral row tilt; per-element radial densities reveal core–shell chemistry (cation vs anion radius); nearest-neighbour and coordination distributions pin the bonding motif; axial folding and linear density describe stacking along the wire.

### Suggested next steps
- **Chiral index extraction** — fit the 2-D lattice on the unrolled map to assign $(n,m)$-style indices per tube.
- **Curvature-energy correlation** — regress `e_above_hull` / `e_form` against $r_{95}$ and $1/r$ to expose the strain–stability trade-off.
- **Voronoi/CrystalNN coordination** (in a pymatgen env) for bond topology beyond a distance cutoff.
- **Symmetry-aware deduplication** — cluster tubes by the rotation/translation-invariant radial fingerprint (cf. `reduce_templates.fingerprint`) to find near-duplicate templates.
- **Composition maps** — $r_{95}$, wall count and ellipticity vs chemistry (period/group of the constituent elements).

## 10 · Six random unit-cell examples

Shared with the synthetic-set notebook: detailed metrics + cross-sections for 6 randomly drawn tubes.

In [ ]:
# --- 6 random unit-cell examples: detailed per-tube metrics -------------------
# `tube_metrics` is the single per-tube descriptor reused by the quality filter.
def tube_metrics(t):
    "Per-tube geometry/chemistry descriptor from cart + cell + elements + frac."
    cart = np.asarray(t["cart"], float); cell = np.asarray(t["cell"], float)
    els  = np.asarray(t["elements"]); n = len(cart)
    ax   = detect_tube_axis(t["frac"])
    L    = float(np.linalg.norm(cell[ax]))
    r, th, z, u, v = cylindrical_coords(cart, cell, ax)
    r05, r95 = (np.percentile(r, [5, 95]) if r.size else (0.0, 0.0))  # mask-matching edges
    ctr, rho, cnt = radial_density(r, L, nbins=24)
    pos = rho[rho > 0]
    rho_peak = float(rho.max()) if rho.size else 0.0
    rho_mean = float(pos.mean()) if pos.size else 0.0
    peak_ratio = (rho_peak / rho_mean) if rho_mean > 0 else 0.0
    walls = max(1, len(find_peaks(np.concatenate([[0], rho, [0]]))[0]))
    nn = nn_distances(cart, cell, ax).ravel() if n > 1 else np.array([])
    cn, _ = coordination(cart, cell, ax) if n > 1 else (np.array([]), None)
    eps = ellipticity(u, v)[0] if n >= 3 else np.nan
    return dict(
        formula=t.get("formula"), nsites=int(n), axis="abc"[ax], L=L,
        r_min=float(r05), r_max=float(r95),
        r_min_abs=float(r.min()) if r.size else np.nan,
        r_max_abs=float(r.max()) if r.size else np.nan,
        wall_thickness=float(r95 - r05), wall_std=float(r.std()) if r.size else np.nan,
        walls=int(walls), peak_ratio=float(peak_ratio),
        min_nn=float(nn.min()) if nn.size else np.nan,
        median_bond=float(np.median(nn)) if nn.size else np.nan,
        mean_cn=float(np.mean(cn)) if len(cn) else np.nan,
        ellipticity=float(eps) if eps == eps else np.nan,
        lin_dens=n / L if L > 0 else np.nan,
    )

_rngA = np.random.default_rng(SEED + 1)
_pick = _rngA.choice(len(templates), size=min(6, len(templates)), replace=False)
examples = [templates[i] for i in _pick]

print(f"6 random unit-cell examples (seed={SEED + 1})\n" + "=" * 74)
for t in examples:
    m = tube_metrics(t)
    raw = f"  [raw n={t['nsites_raw']}, {t.get('reduce_tag', '')}]" if "nsites_raw" in t else ""
    print(f"\n─ {m['formula']}  (n={m['nsites']}, axis={m['axis']}, L={m['L']:.2f} Å){raw}")
    print(f"   radius   r_min(5%)={m['r_min']:.2f}  r_max(95%)={m['r_max']:.2f} Å"
          f"   (abs {m['r_min_abs']:.2f}–{m['r_max_abs']:.2f})")
    print(f"   wall     thickness={m['wall_thickness']:.2f} Å  std={m['wall_std']:.2f} Å  "
          f"walls≈{m['walls']}  ρ_peak/ρ_mean={m['peak_ratio']:.2f}")
    print(f"   bonding  min contact={m['min_nn']:.2f} Å  median bond={m['median_bond']:.2f} Å  "
          f"⟨CN⟩={m['mean_cn']:.2f}")
    print(f"   shape    ellipticity={m['ellipticity']:.2f}  linear density={m['lin_dens']:.2f} atoms/Å")

# cross-section small-multiples (theta, r) coloured by element
fig, axes = plt.subplots(2, 3, figsize=(12, 8), subplot_kw=dict(projection="polar"))
for a, t in zip(axes.flat, examples):
    ax_ = detect_tube_axis(t["frac"])
    r, th, z, u, v = cylindrical_coords(np.asarray(t["cart"], float),
                                        np.asarray(t["cell"], float), ax_)
    els = np.asarray(t["elements"]); cmap = elem_colors(els)
    for el in sorted(set(els)):
        mm = els == el
        a.scatter(th[mm], r[mm], s=20, color=cmap[el], label=str(el),
                  edgecolor="white", linewidth=0.3)
    a.set_title(f"{t.get('formula')}  (n={len(t['cart'])})", fontsize=9, pad=8)
    a.set_xticklabels([]); a.set_yticklabels([])
    a.legend(fontsize=6, loc="upper right", bbox_to_anchor=(1.30, 1.15),
             handletextpad=0.1, borderpad=0.1, framealpha=0.6)
for k in range(len(examples), 6):
    axes.flat[k].axis("off")
fig.suptitle("6 random unit-cell examples — cross-section (θ, r) by element",
             y=1.01, fontweight="bold")
plt.tight_layout(); plt.show()


### 3D perspective views of the 6 example tubes

Same 6 tubes as above, in perspective. Each panel marks **r_min** (blue cylinder), **r_max** (red cylinder) about the detected tube axis, and the atoms' **x / y / z extents in Å** on the bounding box.

In [ ]:
# --- 3D perspective views of the 6 example tubes -----------------------------
# Reuses `examples` from the cell above. r_min/r_max cylinders follow the tube
# axis; x/y/z labels give the Cartesian bounding-box extent in Ångström.
import itertools
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (registers 3d projection)

def _tube_frame(t):
    "Return (cart, elements, a_hat, e1, e2, ctr, r, z) in the tube-axis frame."
    cart = np.asarray(t["cart"], float); cell = np.asarray(t["cell"], float)
    els  = np.asarray(t["elements"])
    ax_i = detect_tube_axis(t["frac"])
    a_hat = cell[ax_i] / np.linalg.norm(cell[ax_i])
    z = cart @ a_hat
    perp = cart - np.outer(z, a_hat)
    other = [k for k in range(3) if k != ax_i]
    e1 = cell[other[0]] - (cell[other[0]] @ a_hat) * a_hat
    e1 = e1 / np.linalg.norm(e1)
    e2 = np.cross(a_hat, e1)
    ctr = perp.mean(0)
    r = np.hypot((perp - ctr) @ e1, (perp - ctr) @ e2)
    return cart, els, a_hat, e1, e2, ctr, r, z

def draw_tube_3d(ax, t):
    cart, els, a_hat, e1, e2, ctr, r, z = _tube_frame(t)
    r_min, r_max = (np.percentile(r, [5, 95]) if r.size else (0.0, 0.0))
    z0, z1 = (z.min(), z.max()) if z.size else (0.0, 1.0)
    cmap = elem_colors(els)
    for el in sorted(set(els)):                       # atoms, coloured by element
        m = els == el
        ax.scatter(cart[m, 0], cart[m, 1], cart[m, 2], s=45, color=cmap[el],
                   edgecolors="k", linewidths=0.3, label=str(el), depthshade=True)
    th = np.linspace(0, 2 * np.pi, 48)                # r_min / r_max cylinders + end rings
    TZ = np.array([z0, z1])
    for R, col in [(r_min, "#277DA1"), (r_max, "#F94144")]:
        TT, ZZ = np.meshgrid(th, TZ)
        S = (ctr[None, None, :] + ZZ[..., None] * a_hat[None, None, :]
             + R * (np.cos(TT)[..., None] * e1[None, None, :]
                    + np.sin(TT)[..., None] * e2[None, None, :]))
        ax.plot_surface(S[..., 0], S[..., 1], S[..., 2], color=col, alpha=0.12,
                        linewidth=0, shade=False)
        ring = ctr + z1 * a_hat + R * (np.cos(th)[:, None] * e1 + np.sin(th)[:, None] * e2)
        ax.plot(ring[:, 0], ring[:, 1], ring[:, 2], color=col, lw=1.6)
    lo, hi = cart.min(0), cart.max(0); d = hi - lo    # bounding box + x/y/z Å labels
    corners = np.array(list(itertools.product(*zip(lo, hi))))
    for i, j in itertools.combinations(range(len(corners)), 2):
        if int((corners[i] != corners[j]).sum()) == 1:
            seg = np.vstack([corners[i], corners[j]])
            ax.plot(seg[:, 0], seg[:, 1], seg[:, 2], color="0.6", lw=0.5, alpha=0.5)
    ax.text((lo[0] + hi[0]) / 2, lo[1], lo[2], f"x = {d[0]:.1f} Å", fontsize=8, color="k")
    ax.text(hi[0], (lo[1] + hi[1]) / 2, lo[2], f"y = {d[1]:.1f} Å", fontsize=8, color="k")
    ax.text(hi[0], lo[1], (lo[2] + hi[2]) / 2, f"z = {d[2]:.1f} Å", fontsize=8, color="k")
    try:
        ax.set_box_aspect(tuple(np.maximum(d, 1e-3)))
    except Exception:
        pass
    ax.view_init(elev=18, azim=-60)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    ax.set_title(f"{t.get('formula')}\n"
                 f"r$_{{min}}$={r_min:.2f}  r$_{{max}}$={r_max:.2f} Å", fontsize=9)
    ax.legend(fontsize=6, loc="upper left", handletextpad=0.1, borderpad=0.1,
              framealpha=0.6)

fig = plt.figure(figsize=(15, 9.5))
for k, t in enumerate(examples):
    ax = fig.add_subplot(2, 3, k + 1, projection="3d")
    draw_tube_3d(ax, t)
fig.suptitle("3D perspective — atoms, r$_{min}$ (blue) / r$_{max}$ (red) cylinders, "
             "and x/y/z extents (Å)", y=0.99, fontweight="bold")
plt.tight_layout(); plt.show()


## 11 · Quality filter — better templates for the generation mask

Population pass over all 13,295 tubes; funnel of survivors per gate. The real set is DFT-relaxed, so it should survive far better than the synthetic stage-0 set.

In [ ]:
# --- Quality filter: pick better templates for the generation mask ------------
# Gates select templates that give the `shl` mask a physical, guidance-friendly
# geometry. Tune the constants below; the funnel shows how many survive each gate.
MIN_CONTACT = 0.7     # Å  minimum nearest-neighbour distance (reject unphysical overlaps)
MIN_WALL    = 0.3     # Å  r_max-r_min (reject 1D chains / point cross-sections)
MIN_RADIUS  = 1.0     # Å  r_max (95th pct) must be a real tube radius
NATM_MIN    = 4       # atoms  lower bound (need a real 2D cross-section)
NATM_MAX    = 128     # atoms  DB reduction cap
PEAK_RATIO  = 2.0     # ρ_peak/ρ_mean (density-guidance viability; flat ρ -> no gradient)
SINGLE_WALL = False   # if True, also require walls == 1

# Compute metrics once over the whole population (reused if already built above).
if "M" not in globals() or len(M) != len(templates):
    _t0 = time.time()
    M = pd.DataFrame([tube_metrics(t) for t in templates])
    print(f"Computed metrics for {len(M):,} templates in {time.time() - _t0:.1f}s")

gates = [
    (f"contacts   (min_nn >= {MIN_CONTACT:.2f} Å)", M.min_nn >= MIN_CONTACT),
    (f"wall       (r_max-r_min >= {MIN_WALL:.2f} Å)", M.wall_thickness >= MIN_WALL),
    (f"radius     (r_max >= {MIN_RADIUS:.2f} Å)", M.r_max >= MIN_RADIUS),
    (f"atoms      ({NATM_MIN} <= n <= {NATM_MAX})", (M.nsites >= NATM_MIN) & (M.nsites <= NATM_MAX)),
    (f"peaked ρ   (ρ_peak/ρ_mean >= {PEAK_RATIO:.1f})", M.peak_ratio >= PEAK_RATIO),
]
if SINGLE_WALL:
    gates.append(("single-wall (walls == 1)", M.walls == 1))

keep = np.ones(len(M), bool)
print(f"\n{'gate':42s} {'pass':>8s} {'cumulative':>11s}")
print("-" * 63)
for name, mask in gates:
    mask = mask.fillna(False).values
    keep &= mask
    print(f"{name:42s} {int(mask.sum()):8d} {int(keep.sum()):11d}")
print("-" * 63)
n_keep = int(keep.sum())
print(f"SURVIVING TEMPLATES: {n_keep:,} / {len(M):,}  ({100 * n_keep / max(len(M), 1):.1f}%)")

removed = {name: int((~mask.fillna(False)).sum()) for name, mask in gates}
worst = max(removed, key=removed.get)
print(f"Most-culling gate (standalone): '{worst.strip()}' removes {removed[worst]:,} templates")

M["passes"] = keep
_cols = ["r_min", "r_max", "wall_thickness", "peak_ratio", "min_nn", "nsites"]
print("\nSummary of surviving templates:")
display(M[keep][_cols].describe().round(2))
